# Colab Minimal CFM Overfit Guide

## Recommended: minimal CFM overfit (no Stage 2, no author `results/`)

Run these cells in order:

| Step | Cell | What it does | Time (rough) |
|------|------|--------------|--------------|
| 1 | **1** | Enter project dir | < 1 min |
| 2 | **2** | Install dependencies | 2–5 min |
| 3 | **3** | WandB login (optional) | 1 min |
| 4 | **4** | Check `Data/` exists | < 1 min |
| 5 | **6** | Stage 1 smoke: train GRU 5 epochs | 5–20 min |
| 6 | **8** | Resolve `extractor_path` from latest Stage 1 run | < 1 min |
| 7 | **11** | Stage 3 CFM overfit 100 epochs (1 sample) | 30–90 min |

**Skip:** Cell 5 (one-click), Cell 7 (Stage 2), Cell 9/10 (smoke tests)

## Condition used in Cell 11
`c = [hidden_state, direction, DI]`
- `hidden_state`: from Stage 1 GRU checkpoint (Cell 6 output)
- `direction`, `DI`: from `Data/DamagePattern/label.csv` (no Stage 2 needed)

## If you already have author checkpoints
Skip Cell 6 and run **1 → 2 → 4 → 8 → 11** instead.
Upload:
- `results/HystereticPrediction/HystereticGRU/2024-06-12_18-13-47`

## Success check
- Log shows: `[Overfit mode] using one sample only`
- `train/fm_loss` keeps decreasing
- At epoch 100: `val/psnr` / `val/ssim` should improve


In [ ]:
# === EDIT THIS IF NEEDED ===
PROJECT_DIR = "/content/intership-project"

import os
assert os.path.isdir(PROJECT_DIR), f"Project dir not found: {PROJECT_DIR}"
%cd /content/intership-project
print("Project dir:", PROJECT_DIR)


In [ ]:
# Install dependencies
!pip install -q --upgrade pip
!pip install -q -r requirements.txt


In [ ]:
# Optional: Weights & Biases
USE_WANDB = True  # set False to disable

if USE_WANDB:
    import wandb
    wandb.login()
    WANDB_FLAG = "--use_wandb"
else:
    WANDB_FLAG = ""

print("WANDB_FLAG:", WANDB_FLAG)


In [ ]:
# Data quick check (must exist before training)
required_paths = [
    "Data/Hysteresis/info_w_mander_big.csv",
    "Data/Hysteresis/id_list_small.json",
    "Data/DamagePattern/label.csv",
    "Data/DamagePattern/id_list.json",
]

for p in required_paths:
    print(p, "OK" if os.path.exists(p) else "MISSING")


In [ ]:
# One-click pipeline runner (GAN / CFM / Overfit)
MODE = "overfit"      # "smoke" or "overfit"
BACKEND = "cfm"     # "gan" or "cfm"

wandb_opt = "--with-wandb" if USE_WANDB else ""
!bash scripts/colab_quickstart.sh {wandb_opt} --mode={MODE} --backend={BACKEND}


In [ ]:
# Stage 1 (manual): Hysteretic prediction
!python train_hysteresis.py --architecture HystereticGRU --epoch 5 --batch_size 4 --num_workers 1 {WANDB_FLAG}


In [ ]:
# Stage 2 (manual): Damage quantification
!python train_quantification.py --backbone vit_b_16 --epoch 5 --batch_size 4 --num_workers 1 {WANDB_FLAG}


In [ ]:
# Resolve Stage 1 extractor path (minimal CFM path: Stage 2 not required)
# Also check whether train[0] and val[0] are the same filename in id_list.json.
import glob, json

AUTHOR_EXTRACTOR = "results/HystereticPrediction/HystereticGRU/2024-06-12_18-13-47"

def latest_run(base_glob):
    runs = sorted(glob.glob(base_glob), key=os.path.getmtime)
    assert runs, f"No runs found for: {base_glob}. Run Cell 6 first."
    return runs[-1]

if os.path.isdir(AUTHOR_EXTRACTOR):
    extractor_path = AUTHOR_EXTRACTOR
    print("Using author extractor:", extractor_path)
else:
    extractor_path = latest_run("results/HystereticPrediction/HystereticGRU/*")
    print("Using latest local Stage 1 run:", extractor_path)

with open("Data/DamagePattern/id_list.json", "r") as f:
    split = json.load(f)
train0 = split["train"][0]
val0 = split["val"][0]
print("train[0] =", train0)
print("val[0]   =", val0)
print("same file?", train0 == val0)
print("Note: overfit mode in train_mixed.py now forces train/val to use the SAME train sample.")


In [ ]:
# Stage 3 (manual): GAN backend smoke
!python train_mixed.py \
  --architecture HystereticCGAN \
  --loss_type SGAN \
  --extractor_path "{extractor_path}" \
  --regressor_path "{regressor_path}" \
  --data_path id_list.json \
  --latent_conditioning False \
  --direction_conditioning False \
  --add_DI_loss False \
  --image_size 128 \
  --z_dim 100 \
  --G_h_size 128 \
  --D_h_size 128 \
  --SELU False \
  --spectral_D False \
  --no_batch_norm False \
  --lr_schedule warmup_cosine \
  --warmup_epochs 2 \
  --min_lr_ratio 0.1 \
  --epoch 5 \
  --batch_size 4 \
  --num_workers 1 \
  {WANDB_FLAG}


In [ ]:
# Stage 3 (manual): CFM ADM backend with Heun solver
!python train_mixed.py \
  --architecture CFMADM \
  --cfm_solver heun \
  --cfm_ode_steps 20 \
  --cfm_model_channels 128 \
  --cfm_num_res_blocks 2 \
  --cfm_cond_dim 512 \
  --cfm_dropout 0.0 \
  --extractor_path "{extractor_path}" \
  --regressor_path "{regressor_path}" \
  --data_path id_list.json \
  --latent_conditioning True \
  --direction_conditioning True \
  --include_di_conditioning True \
  --add_DI_loss False \
  --image_size 128 \
  --lr_schedule warmup_cosine \
  --warmup_epochs 2 \
  --min_lr_ratio 0.1 \
  --epoch 5 \
  --batch_size 4 \
  --num_workers 1 \
  {WANDB_FLAG}


In [ ]:
# Minimal path: Stage 3 CFM overfit (100 epochs, single sample, no Stage 2)
!python train_mixed.py \
  --architecture CFMADM \
  --cfm_solver heun \
  --cfm_ode_steps 50 \
  --extractor_path "{extractor_path}" \
  --data_path id_list.json \
  --latent_conditioning True \
  --direction_conditioning True \
  --include_di_conditioning True \
  --add_DI_loss False \
  --image_size 128 \
  --lr_schedule warmup_cosine \
  --warmup_epochs 10 \
  --min_lr_ratio 0.1 \
  --overfit_single_sample \
  --overfit_sample_index 0 \
  --overfit_repeat_steps 32 \
  --epoch 100 \
  --batch_size 4 \
  --num_workers 0 \
  {WANDB_FLAG}


In [ ]:
# Stage 3 result visualization: GT vs CFM inference (from latest CFM run)
import os, glob, json, argparse, importlib
import torch
import matplotlib.pyplot as plt
from torchvision import transforms
from src.dataset import PatternLoopDataset


def first_match(pattern):
    matches = glob.glob(pattern, recursive=True)
    assert matches, f"No file found: {pattern}"
    return matches[0]


def latest_cfm_run(base_glob="results/DamagePatternGeneration/*/*"):
    run_dirs = sorted(glob.glob(base_glob), key=os.path.getmtime, reverse=True)
    for run_dir in run_dirs:
        args_candidates = glob.glob(os.path.join(run_dir, "**", "args.json"), recursive=True)
        if not args_candidates:
            continue
        args_data = json.load(open(args_candidates[0], "r"))
        if args_data.get("architecture") == "CFMADM":
            return run_dir
    raise RuntimeError("No CFM run found under results/DamagePatternGeneration/*/*")


class RepeatSingleSampleDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, sample_index=0, repeats=1):
        self.base_dataset = base_dataset
        self.sample_index = sample_index
        self.repeats = repeats

    def __len__(self):
        return self.repeats

    def __getitem__(self, idx):
        return self.base_dataset[self.sample_index]


# 1) Resolve latest CFM run folder
run_dir = latest_cfm_run()
print("CFM run_dir:", run_dir)

# 2) Load saved args and checkpoint
args_path = first_match(os.path.join(run_dir, "**", "args.json"))
run_args = argparse.Namespace(**json.load(open(args_path, "r")))

ckpt_candidates = []
for patt in ["best_fid_*.ckpt", "best_ssim_*.ckpt", "*.ckpt"]:
    ckpt_candidates.extend(glob.glob(os.path.join(run_dir, "**", patt), recursive=True))
assert ckpt_candidates, f"No checkpoint found under {run_dir}"
ckpt_path = sorted(set(ckpt_candidates), key=os.path.getmtime)[-1]
print("Using checkpoint:", ckpt_path)

# 3) Load extractor
extractor_args_path = first_match(os.path.join(run_args.extractor_path, "**", "args.json"))
extractor_args = argparse.Namespace(**json.load(open(extractor_args_path, "r")))
extractor = importlib.import_module("src.model.HystereticPrediction").__dict__[extractor_args.architecture](**vars(extractor_args))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
extractor = extractor.to(device)
extractor_ckpt = first_match(os.path.join(run_args.extractor_path, "**", "best.ckpt"))
extractor.load_state_dict(torch.load(extractor_ckpt, map_location=device)["state_dict"])
extractor.eval()

# 4) Build dataset and take same overfit sample
transform = transforms.Compose([
    transforms.Resize((run_args.image_size, run_args.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

base_train_ds = PatternLoopDataset(
    root="./Data",
    mode="train",
    transform=transform,
    extractor=extractor,
    extractor_args=extractor_args,
    **vars(run_args),
)

sample_idx = int(getattr(run_args, "overfit_sample_index", 0))
overfit_ds = RepeatSingleSampleDataset(base_train_ds, sample_index=sample_idx, repeats=1)
real_data, hidden_state, direction, exp, timestep, DI, info = overfit_ds[0]

real_data = real_data.unsqueeze(0).to(device)
hidden_state = hidden_state.unsqueeze(0).to(device)
direction = direction.unsqueeze(0).to(device)
DI = DI.unsqueeze(0).to(device)
info = info.unsqueeze(0).to(device)

# 5) Load CFM model and sample
model = importlib.import_module("src.model.ImageGeneration").__dict__[run_args.architecture](regressor=None, **vars(run_args))
model = model.to(device)
model.load_state_dict(torch.load(ckpt_path, map_location=device)["state_dict"], strict=False)
model.eval()

with torch.no_grad():
    c = model.process_condition(info, hidden_state, direction, DI)
    pred = model.sample_with_ode(c, num_steps=50)

real_vis = torch.clamp(real_data * 0.5 + 0.5, 0.0, 1.0)[0].permute(1, 2, 0).cpu().numpy()
pred_vis = torch.clamp(pred * 0.5 + 0.5, 0.0, 1.0)[0].permute(1, 2, 0).cpu().numpy()

# 6) Plot GT vs inference
fig = plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(real_vis)
plt.title("GT")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(pred_vis)
plt.title("CFM Inference")
plt.axis("off")

plt.suptitle(f"Exp: {exp} | timestep: {timestep} | sample_idx: {sample_idx}")
plt.tight_layout()
plt.show()

# 7) Save side-by-side figure
save_path = os.path.join(run_dir, "gt_vs_inference.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved figure to:", save_path)